# Appendix F: Under the Hood

How LLMs actually work: tokenization, attention, the KV cache, and sampling.

This notebook is an original tutorial (rewritten — the previous version
described a folk model of generation that is not how transformers work).

## Learning Objectives

- Trace one token's journey: text → tokens → embeddings → transformer blocks → logits → sampled token.
- Explain attention in one paragraph and why context length costs what it costs.
- Explain the KV cache — the single concept that connects internals to API pricing and latency.
- Use sampling parameters (temperature, top-p) with an accurate mental model.

## Mental Model

Generation is a loop around one function: **given all tokens so far, produce
a probability distribution over the next token; sample one; append; repeat.**

```
"The invoice is " ─tokenize─▶ [791, 25637, 374, 220]
      │ each step:
      ▼
embeddings ─▶ N transformer blocks (attention + MLP) ─▶ logits (one per vocab token)
      ▼
softmax(logits / temperature) ─▶ top-p filter ─▶ sample ─▶ "overdue"
      └────────────── append, repeat ──────────────┘
```

There is **no candidate-evaluation stage, no revision pass, no lookahead** —
one forward pass per token, greedy accumulation. Everything that looks like
deliberation (chain-of-thought, self-correction, "thinking") is *more tokens
through the same loop*, not a different mechanism. That single fact
demystifies most LLM behavior.

## Important Concepts

- **Tokenization (BPE)**: text is split into subword units from a fixed vocabulary (~50k-200k). Why models are bad at counting letters and arithmetic on digits: they see tokens, not characters. "strawberry" may be 1-3 tokens; "1234567" splits unpredictably.
- **Embeddings**: each token id maps to a learned vector; position information is added (rotary embeddings in modern models).
- **Attention**: each token computes query/key/value vectors; its output is a weighted mix of *all previous tokens'* values, weighted by query-key similarity. This is how token 5000 can depend on token 3. Cost grows with context length — the physical reason long context is expensive.
- **Transformer block**: attention (mix information across positions) + MLP (transform each position). Stack N of them (dozens in production models). MoE variants route each token through a subset of expert MLPs (why "671B total / 37B active" — notebook 30).
- **Logits → sampling**: the final layer scores every vocab token. `temperature` divides logits before softmax (0 → argmax-ish; higher → flatter distribution); `top-p` samples only from the smallest set of tokens whose probability sums to p.
- **KV cache**: at each step, keys/values for all previous tokens are *reused*, not recomputed — only the new token runs the full stack. Consequences: prefill (processing the prompt) vs decode (per-token generation) split; **prompt caching** = persisting the KV cache of a stable prefix across requests (the 50-90% savings in notebook 27); long context costs GPU memory because the cache lives there.

## Need-To-Know Coverage Checklist

- [x] BPE tokenization and its behavioral consequences (counting, arithmetic).
- [x] Attention as query-key-weighted value mixing; context-length cost.
- [x] Transformer block anatomy; MoE in one line.
- [x] Logits, temperature, top-p — accurate mechanics.
- [x] KV cache → prefill/decode split, prompt caching, memory cost of context.
- [x] Corrected folk model: no candidate-evaluation or revision pass exists.

## Deep Study Notes

### Why the folk model matters

A common wrong story: "the model drafts candidate responses, evaluates them,
and refines the best one." A standard forward pass does none of that. The
accurate story — one distribution per step, sampled greedily forward —
explains real phenomena the folk model can't:

- **Early tokens constrain everything after** (no revision pass exists);
  this is why output *format* choices ("start with the answer") change quality.
- **Hallucinations are fluent** — the model samples plausible continuations;
  there is no fact-checking stage to catch them.
- **Reasoning models** work by generating *more tokens* (hidden thinking)
  before the visible answer — same loop, more compute — not by a secret
  deliberation mechanism (notebook 17).

### The KV cache — the interview connector

Each generated token needs attention over all previous tokens' keys/values.
Recomputing those every step would be quadratic-per-response; caching them
makes each step cheap but *stateful*:

- **Prefill** (prompt processing) is compute-bound and parallel; **decode**
  (token-by-token) is memory-bandwidth-bound and sequential. This is why
  time-to-first-token and tokens-per-second are quoted separately (notebook 23's
  latency budget).
- **Prompt caching** persists the prefix's KV cache across requests: system
  prompt + tools + interview guide never re-prefill. That is mechanically
  where the 50-90% cost cut in notebook 27 comes from — and why caching
  requires *exact* prefix match.
- Context limits are substantially **GPU-memory limits on the cache**.

### Sampling, precisely

`softmax(logits / T)`: T<1 sharpens (more deterministic), T>1 flattens.
Top-p keeps the smallest token set with cumulative probability ≥ p.
Temperature 0 still isn't reproducible on hosted APIs — batch non-invariance
and MoE routing (notebook 22) — a chain of concepts that plays very well in
interviews: tokenizer → logits → sampling → why temp-0 is still
non-deterministic.

## Common Failure Modes

- Repeating the candidate-evaluation folk model in an interview → signals no internals knowledge.
- Expecting character-level skills (count the r's) from a token-level model.
- Prompt "caching" with a prefix that varies per request (timestamps in the system prompt) → zero hits.
- Treating temperature as a "creativity" dial rather than distribution sharpening → cargo-cult tuning.
- Assuming long context is cheap because it fits → prefill cost and cache memory scale with it.

## Implementation Notes

- Put stable content first, variable content last — that's the whole art of cache-friendly prompts.
- When output quality varies wildly run-to-run, check temperature and top-p before blaming the prompt.
- For token-sensitive logic (budgets, truncation) use the model's real tokenizer, not word counts.

In [ ]:
"""The generation loop, faithfully miniaturized: tokens -> logits ->
temperature/top-p sampling -> append -> repeat, with a toy KV cache
demonstrating why prefix reuse saves money.
"""
import math
import random

VOCAB = ["overdue", "paid", "pending", "banana"]


def fake_logits(context_tokens):
    """Stands in for N transformer blocks: context -> score per vocab token."""
    base = {"overdue": 3.0, "paid": 2.2, "pending": 1.5, "banana": -2.0}
    return [base[t] for t in VOCAB]


def sample(logits, temperature=0.8, top_p=0.9, rng=None):
    if temperature == 0:
        return VOCAB[logits.index(max(logits))]
    scaled = [l / temperature for l in logits]
    exps = [math.exp(l - max(scaled)) for l in scaled]
    probs = [e / sum(exps) for e in exps]
    # top-p: keep the smallest set of tokens with cumulative prob >= p
    order = sorted(range(len(VOCAB)), key=lambda i: -probs[i])
    kept, cum = [], 0.0
    for i in order:
        kept.append(i); cum += probs[i]
        if cum >= top_p:
            break
    kept_probs = [probs[i] for i in kept]
    total = sum(kept_probs)
    return VOCAB[rng.choices(kept, [p / total for p in kept_probs])[0]]


rng = random.Random(0)
for temp in (0, 0.8, 2.0):
    outs = [sample(fake_logits(["the", "invoice", "is"]), temp, rng=rng or random.Random())
            if temp else sample(fake_logits([]), 0) for _ in range(8)]
    print(f"T={temp}: {outs}")
# T=0 always argmax; T=0.8 mostly 'overdue'; T=2.0 nearly uniform over kept set.

# --- KV cache economics ----------------------------------------------------
def cost_without_cache(prompt_len, gen_len):
    # every step reprocesses the whole prefix
    return sum(prompt_len + i for i in range(gen_len))

def cost_with_cache(prompt_len, gen_len, prefix_cached=0):
    prefill = prompt_len - prefix_cached      # only uncached prefix prefilled
    return prefill + gen_len                  # then one unit per new token

P, G = 5000, 200
print(f"\nno KV cache:          {cost_without_cache(P, G):>9,} token-passes")
print(f"KV cache:             {cost_with_cache(P, G):>9,}")
print(f"KV + prompt caching:  {cost_with_cache(P, G, prefix_cached=4500):>9,}"
      "  <- why stable prefixes cut cost 50-90%")


## Practice

1. Set top_p=0.5 at T=2.0 and explain why the output tightens even though
   temperature is high — which knob dominated?
2. Using the cost functions, compute the savings for the notebook-27 voice
   agent: 3k-token stable prefix (system + guide), 2k growing transcript,
   150-token turns.
3. Explain to a PM why the model can write a sonnet but miscounts the r's in
   "strawberry" — in exactly two sentences, using the word "token".
4. Chain the concepts for an interviewer: tokenizer → logits → temperature →
   why temperature-0 outputs still differ across runs (bring in notebook 22).

## Design Checklist

- [ ] Mental model has no candidate-evaluation/revision stage.
- [ ] Prompts structured stable-prefix-first for cache hits.
- [ ] Token counting done with a real tokenizer.
- [ ] Sampling params set deliberately per task (extraction: low T; ideation: higher T + top-p).
- [ ] Long-context costs (prefill, cache memory) budgeted, not assumed free.